# Job Market Trends - Enhanced ETL Pipeline

This notebook implements a structured ETL pipeline that integrates job postings with company metrics, industries, and **skills**. It includes stage-wise logging and data quality tracking.

In [ ]:
import os
import datetime
import pandas as pd
import numpy as np
import kagglehub

class StepLogger:
    def __init__(self):
        self.logs = []
    
    def log(self, step_name, details, df=None):
        row_count = len(df) if df is not None else "N/A"
        entry = {
            'Timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'Step': step_name,
            'Details': details,
            'Row Count': row_count
        }
        self.logs.append(entry)
        print(f"[{entry['Timestamp']}] {step_name}: {details} (Rows: {row_count})")

    def get_summary(self):
        return pd.DataFrame(self.logs)

logger = StepLogger()

## 1. Data Ingestion

In [ ]:
def load_datasets():
    logger.log("Ingestion", "Locating dataset from Kaggle")
    data_path = kagglehub.dataset_download("arshkon/linkedin-job-postings")
    
    datasets = {
        'jobs': pd.read_csv(os.path.join(data_path, 'postings.csv')),
        'companies': pd.read_csv(os.path.join(data_path, 'companies', 'companies.csv')),
        'employees': pd.read_csv(os.path.join(data_path, 'companies', 'employee_counts.csv')),
        'industries': pd.read_csv(os.path.join(data_path, 'companies', 'company_industries.csv')),
        'ind_labels': pd.read_csv(os.path.join(data_path, 'mappings', 'industries.csv')),
        'job_skills': pd.read_csv(os.path.join(data_path, 'jobs', 'job_skills.csv')),
        'skills': pd.read_csv(os.path.join(data_path, 'mappings', 'skills.csv'))
    }
    
    for name, df in datasets.items():
        logger.log("Ingestion", f"Loaded {name}", df=df)
    
    return datasets

data_bundle = load_datasets()

## 2. Merging Datasets

In [ ]:
def merge_data(bundle):
    jobs = bundle['jobs']
    companies = bundle['companies']
    employees = bundle['employees']
    industries = bundle['industries']
    ind_labels = bundle['ind_labels']
    job_skills = bundle['job_skills']
    skills = bundle['skills']

    # 1. Join Jobs with Companies
    df = jobs.merge(companies[['company_id', 'name', 'company_size']], left_on='company_id', right_on='company_id', how='left')
    logger.log("Merge", "Joined jobs with company metadata", df=df)

    # 2. Join with Latest Employee Counts
    latest_employees = employees.sort_values('time_recorded').groupby('company_id').tail(1)
    df = df.merge(latest_employees[['company_id', 'employee_count']], on='company_id', how='left')
    logger.log("Merge", "Joined with latest employee counts", df=df)

    # 3. Join with Industries
    industry_full = industries.merge(ind_labels, on='industry_id', how='left')
    comp_industries = industry_full.groupby('company_id')['industry_name'].apply(lambda x: ', '.join(x)).reset_index()
    df = df.merge(comp_industries, on='company_id', how='left')
    logger.log("Merge", "Joined with company industries", df=df)
    
    # 4. Join with Skills
    skills_full = job_skills.merge(skills, on='skill_abr', how='left')
    job_skills_summary = skills_full.groupby('job_id')['skill_name'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    df = df.merge(job_skills_summary, on='job_id', how='left')
    logger.log("Merge", "Joined with job skills", df=df)

    return df

merged_df = merge_data(data_bundle)

## 3. Cleaning & Validation

In [ ]:
def clean_data(df):
    df = df.drop_duplicates()
    
    # Type Conversions
    numeric_cols = ['views', 'applies', 'min_salary', 'max_salary', 'normalized_salary', 'employee_count']
    for col in numeric_cols:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')

    time_cols = ['listed_time', 'expiry', 'original_listed_time']
    for col in time_cols:
        if col in df.columns: df[col] = pd.to_datetime(df[col], errors='coerce', unit='ms')

    text_cols = ['title', 'location', 'name', 'industry_name', 'skill_name', 'formatted_work_type', 'formatted_experience_level']
    for col in text_cols:
        if col in df.columns: df[col] = df[col].astype('string').str.strip().replace(['nan', 'None'], pd.NA)

    logger.log("Cleaning", "Cleaned types and standardized strings", df=df)
    return df

cleaned_df = clean_data(merged_df)

## 4. Feature Engineering

In [ ]:
def transform_data(df):
    df['year'] = df['listed_time'].dt.year
    df['month'] = df['listed_time'].dt.month
    
    df['remote_allowed'] = df['remote_allowed'].fillna(0)
    df['is_remote'] = df['remote_allowed'].map({1.0: 'Remote', 0.0: 'Onsite'})
    
    df['views'] = df['views'].fillna(0)
    df['applies'] = df['applies'].fillna(0)
    df['demand_score'] = df['views'] + (df['applies'] * 2)

    def group_exp(x):
        if pd.isna(x): return 'Not Specified'
        x = str(x).lower()
        if 'senior' in x or 'director' in x or 'executive' in x: return 'Senior'
        if 'entry' in x or 'intern' in x: return 'Entry'
        return 'Mid-level'
    
    df['experience_group'] = df['formatted_experience_level'].apply(group_exp)
    df['final_company_name'] = df['name'].fillna(df['company_name']).fillna('Unknown')
    
    logger.log("Transformation", "Generated features and resolved company names", df=df)
    return df

transformed_df = transform_data(cleaned_df)

## 5. Export

In [ ]:
def save_data(df):
    cols = [
        'job_id', 'final_company_name', 'title', 'location', 'industry_name', 
        'skill_name', 'company_size', 'employee_count', 'is_remote', 
        'experience_group', 'normalized_salary', 'demand_score', 'year', 'month'
    ]
    final_df = df[cols].copy()
    final_df.rename(columns={'skill_name': 'top_skills'}, inplace=True)
    final_df.to_csv("linkedin_jobs_dashboard_ready.csv", index=False)
    logger.log("Export", "Saved enriched dataset to CSV with Skills", df=final_df)
    return final_df

final_df = save_data(transformed_df)

In [ ]:
logger.get_summary()